# Clean San Pham Data

Notebook này thực hiện các bước sau:


1. Đọc dữ liệu từ file `sanpham_original.csv` trong thư mục `preprocessed`.

2. Chuẩn hóa các cột:
   - Giá: chuyển về float (nghìn đồng)
   - Giảm giá: chuyển về float (ví dụ 51% → 0.51)
   - Đã bán: chuyển về số lượng bán (ví dụ 33.1K → 33100)
   - Lượt đánh giá, Điểm đánh giá: chuẩn hóa về số, áp dụng các rule logic về dữ liệu thiếu/hỏng

3. Kiểm tra dữ liệu sau xử lý, đảm bảo không còn lỗi logic (rule đánh giá, giá trị null, ký tự thừa, emoji, ...).

4. Thêm cột Seller ID vào sản phẩm bằng cách đối chiếu Shop URL với file shop_cleaned.csv.

5. Làm sạch tên sản phẩm, xóa cột Shop URL nếu cần.

6. Lưu dữ liệu đã xử lý ra file `sanpham_cleaned.csv` trong cùng thư mục

In [22]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Đường dẫn tuyệt đối tới file sanpham_original.csv
input_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_original.csv')
output_path = input_path.parent / 'sanpham_cleaned.csv'

In [23]:
def parse_price_to_thousand(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    digits = re.sub(r'[^0-9]', '', text)
    if not digits:
        return np.nan
    return float(digits) / 1000.0

def parse_discount(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    m = re.search(r'(\d+(?:[\.,]\d+)?)\s*%', text)
    if not m:
        return np.nan
    return float(m.group(1).replace(',', '.')) / 100.0

def parse_sold(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    text = re.sub(r'(?i)\bsold\b', '', text)
    text = text.replace('Đã bán', '').replace('da ban', '')
    text = text.replace(',', '').strip()

    m_k = re.fullmatch(r'(\d+(?:\.\d+)?)\s*[Kk]', text)
    if m_k:
        return int(float(m_k.group(1)) * 1000)

    m_n = re.fullmatch(r'\d+', text)
    if m_n:
        return int(text)

    return np.nan

def parse_review_count(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(',', '.')
    if text == '':
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan

def parse_rating_score(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(',', '.')
    if text == '':
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan

In [24]:
# Đọc dữ liệu từ file gốc
df = pd.read_csv(input_path)

# 1) Giá -> float (nghìn đồng), Giảm giá -> float
if 'Giá' in df.columns:
    df['Giá'] = df['Giá'].apply(parse_price_to_thousand)
if 'Giảm giá' in df.columns:
    df['Giảm giá'] = df['Giảm giá'].apply(parse_discount)

# 2) Chuẩn hóa 2 cột đánh giá
if 'Lượt đánh giá' in df.columns:
    review_count = df['Lượt đánh giá'].apply(parse_review_count)
else:
    review_count = pd.Series(np.nan, index=df.index)
if 'Điểm đánh giá' in df.columns:
    rating_score = df['Điểm đánh giá'].apply(parse_rating_score)
else:
    rating_score = pd.Series(np.nan, index=df.index)

# Rule 1: Nếu Lượt đánh giá null hoặc = 0 thì Điểm đánh giá = null
mask_review_null_or_zero = review_count.isna() | (review_count == 0)
rating_score = rating_score.mask(mask_review_null_or_zero, np.nan)

# Rule 2: Nếu Lượt đánh giá > 0 mà Điểm đánh giá null hoặc = 0 thì Lượt đánh giá = null
mask_review_pos_but_score_bad = (review_count > 0) & (rating_score.isna() | (rating_score == 0))
review_count = review_count.mask(mask_review_pos_but_score_bad, np.nan)

# Áp lại Rule 1 để đồng bộ sau Rule 2
mask_review_null_or_zero = review_count.isna() | (review_count == 0)
rating_score = rating_score.mask(mask_review_null_or_zero, np.nan)

df['Lượt đánh giá'] = review_count
df['Điểm đánh giá'] = rating_score

# 3) Xử lý cột Đã bán
if 'Đã bán' in df.columns:
    df['Đã bán'] = df['Đã bán'].apply(parse_sold).astype('Int64')

# 4) Kiểm tra trước khi lưu
sold_token_left = df['Đã bán'].astype('string').str.contains(r'(?i)sold|Đã bán', regex=True, na=False).sum() if 'Đã bán' in df.columns else 0
rule_1_violation = ((df['Lượt đánh giá'].isna() | (df['Lượt đánh giá'] == 0)) & df['Điểm đánh giá'].notna()).sum()
rule_2_violation = ((df['Lượt đánh giá'] > 0) & (df['Điểm đánh giá'].isna() | (df['Điểm đánh giá'] == 0))).sum()

if sold_token_left > 0:
    raise ValueError(f"Còn {sold_token_left} dòng chưa xử lý xong cột 'Đã bán'")
if rule_1_violation > 0:
    raise ValueError(f"Còn {rule_1_violation} dòng vi phạm Rule 1")
if rule_2_violation > 0:
    raise ValueError(f"Còn {rule_2_violation} dòng vi phạm Rule 2")

# 5) Lưu file đã xử lý
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Output: {output_path}')
print(f'Rows: {len(df)}')
print('Sold token left:', sold_token_left)
print('Rule 1 violations:', rule_1_violation)
print('Rule 2 violations:', rule_2_violation)
df.head()

Output: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_cleaned.csv
Rows: 7027
Sold token left: 0
Rule 1 violations: 0
Rule 2 violations: 0


,ID Sản phẩm,Tên sản phẩm,Danh mục,Giá,Giảm giá,Đã bán,Điểm đánh giá,Lượt đánh giá,Địa điểm,Đường dẫn,Hình ảnh,Tên shop,Shop URL
0,3140194631,🔥100% hàng thật🔥La Roche-Posay Cicaplast Baume...,Chăm Sóc Da,189.140,0.42,156,4.9,28.0,Vietnam,https://www.lazada.vn/products/pdp-i3140194631...,https://img.lazcdn.com/g/p/67ba4e5cd4ccb841708...,AuraBelle,https://www.lazada.vn/shop/oqrz1tvi/?itemId=31...
1,2496648405,Bộ Chăm Sóc Da Hoa Mận LAIKOU Nhật Bản - 5 Sản...,Chăm Sóc Da,56.000,0.23,2800,4.9,1042.0,Vietnam,https://www.lazada.vn/products/pdp-i2496648405...,https://img.lazcdn.com/g/p/36828d8a5434f8c5a65...,LAIKOU.Selection,https://www.lazada.vn/shop/laikou-selection/?i...
2,2784492770,Pores serum Thu Nhỏ Lỗ Chân Lông Úc Tinh Chất ...,Chăm Sóc Da,55.656,0.45,339,5.0,100.0,Vietnam,https://www.lazada.vn/products/pdp-i2784492770...,https://img.lazcdn.com/g/p/b75cb8329d690d4a901...,On Moon,https://www.lazada.vn/shop/on-moon/?itemId=278...
3,3140952505,Bộ chăm sóc da VIBRANT GLAMOUR ANNIES với tone...,Chăm Sóc Da,36.900,0.29,3300,5.0,50.0,Vietnam,https://www.lazada.vn/products/pdp-i3140952505...,https://img.lazcdn.com/g/p/05bda6d439abb88eab3...,ddddVN.Selection,https://www.lazada.vn/shop/ddddvn-selection/?i...
4,1738167488,Serum meiduzi x2 tem vuông phục hồi cấp ẩm - S...,Chăm Sóc Da,70.500,0.14,2600,5.0,343.0,Vietnam,https://www.lazada.vn/products/pdp-i1738167488...,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",Shopđẹp24h chuyên hàng chính hãng,https://www.lazada.vn/shop/shopdep24h-chuyen-h...


In [25]:
# Kiểm tra nhanh sau khi xử lý
check_rule_1 = df[(df['Lượt đánh giá'].isna() | (df['Lượt đánh giá'] == 0)) & df['Điểm đánh giá'].notna()]
check_rule_2 = df[(df['Lượt đánh giá'] > 0) & (df['Điểm đánh giá'].isna() | (df['Điểm đánh giá'] == 0))]
print('Rule 1 violations (phải = 0):', len(check_rule_1))
print('Rule 2 violations (phải = 0):', len(check_rule_2))
df[['Giá', 'Giảm giá', 'Đã bán', 'Điểm đánh giá', 'Lượt đánh giá']].head(10)

Rule 1 violations (phải = 0): 0
Rule 2 violations (phải = 0): 0


,Giá,Giảm giá,Đã bán,Điểm đánh giá,Lượt đánh giá
0,189.140,0.42,156,4.9,28.0
1,56.000,0.23,2800,4.9,1042.0
2,55.656,0.45,339,5.0,100.0
3,36.900,0.29,3300,5.0,50.0
4,70.500,0.14,2600,5.0,343.0
5,30.000,NaN,674,5.0,99.0
6,26.546,0.57,30,5.0,21.0
7,47.000,NaN,211,5.0,16.0
8,119.000,0.40,31,NaN,NaN
9,163.000,0.18,33100,5.0,1988.0


In [26]:
# Đọc file sanpham_cleaned.csv và thay null của cột 'Lượt đánh giá' bằng 0

sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
df_sanpham = pd.read_csv(sanpham_path)

null_before = df_sanpham['Lượt đánh giá'].isna().sum()
df_sanpham['Lượt đánh giá'] = pd.to_numeric(df_sanpham['Lượt đánh giá'], errors='coerce').fillna(0)
null_after = df_sanpham['Lượt đánh giá'].isna().sum()

print(f'Path: {sanpham_path}')
print(f'Null before: {null_before}')
print(f'Null after: {null_after}')
df_sanpham[['Lượt đánh giá', 'Điểm đánh giá']].head(10)

Path: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_cleaned.csv
Null before: 1427
Null after: 0


,Lượt đánh giá,Điểm đánh giá
0,28.0,4.9
1,1042.0,4.9
2,100.0,5.0
3,50.0,5.0
4,343.0,5.0
5,99.0,5.0
6,21.0,5.0
7,16.0,5.0
8,0.0,NaN
9,1988.0,5.0


In [27]:
# Ghi đè lại file sanpham_cleaned.csv sau khi thay null bằng 0 cho 'Lượt đánh giá'
df_sanpham.to_csv(sanpham_path, index=False, encoding='utf-8-sig')

In [28]:
# So sánh Shop URL giữa sanpham_cleaned.csv và shop_cleaned.csv
sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
shop_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/shop_cleaned.csv')

df_sp = pd.read_csv(sanpham_path)
df_shop = pd.read_csv(shop_path)

sp_urls = set(df_sp['Shop URL'].dropna().astype(str).str.strip()) if 'Shop URL' in df_sp.columns else set()
shop_urls = set(df_shop['Shop URL'].dropna().astype(str).str.strip()) if 'Shop URL' in df_shop.columns else set()

common_urls = sp_urls & shop_urls
only_in_sanpham = sp_urls - shop_urls
only_in_shop = shop_urls - sp_urls

print(f'Tổng Shop URL (sanpham_cleaned.csv): {len(sp_urls)}')
print(f'Tổng Shop URL (shop_cleaned.csv): {len(shop_urls)}')
print(f'Trùng nhau: {len(common_urls)}')
print(f'Có trong sanpham_cleaned.csv nhưng không có trong shop_cleaned.csv: {len(only_in_sanpham)}')
print(f'Có trong shop_cleaned.csv nhưng không có trong sanpham_cleaned.csv: {len(only_in_shop)}')

print('\nMẫu URL chỉ có trong sanpham_cleaned.csv:')
print(list(sorted(only_in_sanpham))[:10])

print('\nMẫu URL chỉ có trong shop_cleaned.csv:')
print(list(sorted(only_in_shop))[:10])

Tổng Shop URL (sanpham_cleaned.csv): 7008
Tổng Shop URL (shop_cleaned.csv): 1872
Trùng nhau: 0
Có trong sanpham_cleaned.csv nhưng không có trong shop_cleaned.csv: 7008
Có trong shop_cleaned.csv nhưng không có trong sanpham_cleaned.csv: 1872

Mẫu URL chỉ có trong sanpham_cleaned.csv:
['https://www.lazada.vn/shop/-gia-re-nhat-/?itemId=2636807592&channelSource=pdp', 'https://www.lazada.vn/shop/-gia-re-nhat-/?itemId=584990734&channelSource=pdp', 'https://www.lazada.vn/shop/0wvnp4dx/?itemId=3184284717&channelSource=pdp', 'https://www.lazada.vn/shop/0wvnp4dx/?itemId=3192199815&channelSource=pdp', 'https://www.lazada.vn/shop/183-shop/?itemId=1520443797&channelSource=pdp', 'https://www.lazada.vn/shop/183-shop/?itemId=295786191&channelSource=pdp', 'https://www.lazada.vn/shop/1994s-stores/?itemId=1277481768&channelSource=pdp', 'https://www.lazada.vn/shop/1994s-stores/?itemId=1343482176&channelSource=pdp', 'https://www.lazada.vn/shop/1dbucz65/?itemId=3188868882&channelSource=pdp', 'https://www.la

In [29]:
# Thêm cột Seller ID vào sanpham_cleaned.csv bằng cách đối chiếu Shop URL với shop_cleaned.csv
from urllib.parse import urlparse
from decimal import Decimal, InvalidOperation

sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
shop_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/shop_cleaned.csv')

df_sp = pd.read_csv(sanpham_path)
df_shop = pd.read_csv(shop_path)

def shop_key_from_url(url):
    if pd.isna(url):
        return np.nan
    text = str(url).strip()
    if text == '':
        return np.nan
    try:
        p = urlparse(text)
        # Ví dụ: /shop/oqrz1tvi/ -> key = oqrz1tvi
        parts = [x for x in p.path.strip('/').split('/') if x]
        if len(parts) >= 2 and parts[0].lower() == 'shop':
            return parts[1].lower()
    except Exception:
        pass
    return np.nan

def parse_seller_id(value):
    # Xử lý cả số thường và dạng khoa học: 2.0035E+11 -> 200350000000
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text == '':
        return np.nan
    try:
        dec = Decimal(text)
        return int(dec)
    except (InvalidOperation, ValueError):
        return np.nan

df_sp['_shop_key'] = df_sp['Shop URL'].apply(shop_key_from_url) if 'Shop URL' in df_sp.columns else np.nan
df_shop['_shop_key'] = df_shop['Shop URL'].apply(shop_key_from_url) if 'Shop URL' in df_shop.columns else np.nan
df_shop['Seller ID'] = df_shop['Seller ID'].apply(parse_seller_id).astype('Int64') if 'Seller ID' in df_shop.columns else np.nan

# Mỗi shop_key lấy 1 Seller ID (ưu tiên giá trị không null)
shop_id_map = (
    df_shop
    .dropna(subset=['_shop_key'])
    .sort_values(by=['_shop_key'])
    .drop_duplicates(subset=['_shop_key'], keep='first')[['_shop_key', 'Seller ID']]
    .rename(columns={'Seller ID': 'Seller ID'})
)

df_sp = df_sp.merge(shop_id_map, on='_shop_key', how='left')
df_sp['Seller ID'] = df_sp['Seller ID'].astype('Int64')

matched = df_sp['Seller ID'].notna().sum()
total = len(df_sp)
unmatched = total - matched

# Lưu file mới có thêm Seller ID vào sanpham_cleaned.csv
save_path = sanpham_path
try:
    df_sp.drop(columns=['_shop_key']).to_csv(save_path, index=False, encoding='utf-8-sig')
except PermissionError:
    save_path = sanpham_path.with_name('sanpham_cleaned_new.csv')
    df_sp.drop(columns=['_shop_key']).to_csv(save_path, index=False, encoding='utf-8-sig')
    print('sanpham_cleaned.csv đang bị khóa, đã lưu sang sanpham_cleaned_new.csv')

print(f'Input shop file: {shop_path}')
print(f'Output: {save_path}')
print(f'Tổng dòng: {total}')
print(f'Match Seller ID: {matched}')
print(f'Không match Seller ID: {unmatched}')

df_sp[['Shop URL', 'Seller ID']].head(10)

Input shop file: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\shop_cleaned.csv
Output: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_cleaned.csv
Tổng dòng: 7027
Match Seller ID: 6943
Không match Seller ID: 84


,Shop URL,Seller ID
0,https://www.lazada.vn/shop/oqrz1tvi/?itemId=31...,200983952195
1,https://www.lazada.vn/shop/laikou-selection/?i...,200548320478
2,https://www.lazada.vn/shop/on-moon/?itemId=278...,200164386065
3,https://www.lazada.vn/shop/ddddvn-selection/?i...,200522832214
4,https://www.lazada.vn/shop/shopdep24h-chuyen-h...,1000442453
5,https://www.lazada.vn/shop/saothaiduongpharma/...,200165895015
6,https://www.lazada.vn/shop/the-gioi-lam-dep-sa...,200538448003
7,https://www.lazada.vn/shop/nana-789/?itemId=28...,1000075379
8,NaN,<NA>
9,https://www.lazada.vn/shop/rohto-mentholatum/?...,16440


In [30]:
# Liệt kê các dòng không match Seller ID trong sanpham_with_seller_id.csv
from pathlib import Path
import pandas as pd

sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_with_seller_id.csv')
df_sp = pd.read_csv(sanpham_path)

unmatched_rows = df_sp[df_sp['Seller ID'].isna()].copy()

print(f'Số dòng không match Seller ID: {len(unmatched_rows)}')

cols_show = [c for c in ['ID Sản phẩm', 'Tên sản phẩm', 'Tên shop', 'Shop URL'] if c in unmatched_rows.columns]
unmatched_rows[cols_show].head(50)

Số dòng không match Seller ID: 84


,ID Sản phẩm,Tên sản phẩm,Tên shop,Shop URL
8,2760223087,COMBO FACE_X3 ĐÔNG ANH CHÍNH HÃNG ĐÔNG_ TRÙNG...,NaN,NaN
670,2118700300,Bộ 2 Sữa Dưỡng Thể Ban Đêm NIVEA 8 Super Foods...,NaN,NaN
718,2539829565,Phấn phủ siêu mịn Pond's Magic Translucent Pow...,Heath& Beauty,https://www.lazada.vn/shop/heath-beauty-23/?it...
789,2959710235,Phấn phủ kiềm dầu chống nắng Maybelline New Yo...,Heath& Beauty,https://www.lazada.vn/shop/heath-beauty-23/?it...
1237,3324883664,Máy Xay Cà Phê Điện Toàn Diện Cho Nhà,NaN,NaN
1578,3301058429,Nước Hoa Cindy Đỏ N84 90 ml,Kim Minh beauty care (sản phẩm chăm sóc cá nhân),https://www.lazada.vn/shop/kim-minh-beauty-car...
1660,3300893380,Nước Hoa Thanh Nhã Hương Trà Ô Lan Độc Đáo,NaN,NaN
1784,3211854553,Pure Osmanthus Perfume Women's Long-lasting Li...,NaN,NaN
1857,3284468036,Điện thoại ZTE Nubia Music (4GB/128GB) Màn hìn...,THẮNG THAO MOBILE,https://www.lazada.vn/shop/thang-thao-mobile/?...
1861,2938230289,Điện thoại giá tốt Samsung Galaxy Note 10 ram ...,THẾ GIỚI SMARTPHONE 3979,https://www.lazada.vn/shop/mobile-gia-tot-3979...


In [31]:
# Kiểm tra số giá trị unique trong cột Seller ID của shop_cleaned.csv

shop_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/shop_cleaned.csv')
df_check = pd.read_csv(shop_path)

if 'Seller ID' not in df_check.columns:
    raise KeyError("Không tìm thấy cột 'Seller ID' trong file.")

unique_non_null = df_check['Seller ID'].nunique(dropna=True)
unique_including_null = df_check['Seller ID'].nunique(dropna=False)
null_count = df_check['Seller ID'].isna().sum()

print(f'Nguồn dữ liệu: {shop_path}')
print(f'Số giá trị unique Seller ID (bỏ qua null): {unique_non_null}')
print(f'Số giá trị unique Seller ID (tính cả null): {unique_including_null}')
print(f'Số giá trị null trong Seller ID: {null_count}')

Nguồn dữ liệu: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\shop_cleaned.csv
Số giá trị unique Seller ID (bỏ qua null): 1785
Số giá trị unique Seller ID (tính cả null): 1786
Số giá trị null trong Seller ID: 28


In [32]:
# Đọc file sanpham_cleaned.csv và thay null của cột 'Giảm giá' bằng 0
from pathlib import Path
import pandas as pd

sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_with_seller_id.csv')
df_sp = pd.read_csv(sanpham_path)

null_before = df_sp['Giảm giá'].isna().sum()
df_sp['Giảm giá'] = df_sp['Giảm giá'].fillna(0)
null_after = df_sp['Giảm giá'].isna().sum()

# Thử ghi đè file gốc; nếu bị khóa thì ghi ra file fallback
save_path = sanpham_path
try:
    df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
except PermissionError:
    save_path = sanpham_path.with_name('sanpham_filled_discount.csv')
    df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f'sanpham_with_seller_id.csv đang bị khóa, đã lưu sang {save_path.name}')

print(f'Updated: {save_path}')
print(f'Null in "Giảm giá" before: {null_before}')
print(f'Null in "Giảm giá" after: {null_after}')
df_sp[['Giảm giá']].head(10)

Updated: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_with_seller_id.csv
Null in "Giảm giá" before: 1680
Null in "Giảm giá" after: 0


,Giảm giá
0,0.42
1,0.23
2,0.45
3,0.29
4,0.14
5,0.00
6,0.57
7,0.00
8,0.40
9,0.18


In [33]:
# Xóa cột "Shop URL" trong file sanpham_cleaned.csv
from pathlib import Path
import pandas as pd

sanpham_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
df_sp = pd.read_csv(sanpham_path)

if 'Shop URL' in df_sp.columns:
    df_sp = df_sp.drop(columns=['Shop URL'])
    print('Đã xóa cột "Shop URL".')
    
    # Ghi đè lại file
    save_path = sanpham_path
    try:
        df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
    except PermissionError:
        save_path = sanpham_path.with_name('sanpham_no_shop_url.csv')
        df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f'sanpham_cleaned.csv đang bị khóa, đã lưu sang {save_path.name}')
    
    print(f'Đã cập nhật file: {save_path}')
    print(f'Các cột còn lại: {df_sp.columns.to_list()}')
else:
    print('Không tìm thấy cột "Shop URL".')

Đã xóa cột "Shop URL".
Đã cập nhật file: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_cleaned.csv
Các cột còn lại: ['ID Sản phẩm', 'Tên sản phẩm', 'Danh mục', 'Giá', 'Giảm giá', 'Đã bán', 'Điểm đánh giá', 'Lượt đánh giá', 'Địa điểm', 'Đường dẫn', 'Hình ảnh', 'Tên shop', 'Seller ID']


In [34]:
# Xóa dấu xuống dòng và icon/emoji trong cột "Tên sản phẩm" của file sanpham_cleaned.csv
import re
from pathlib import Path
import pandas as pd

input_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
df_sp = pd.read_csv(input_path)

def clean_product_name(name):
    if not isinstance(name, str):
        return name
    
    # 1. Thay thế nhiều khoảng trắng, tab, xuống dòng bằng một khoảng trắng duy nhất
    cleaned_name = re.sub(r'\s+', ' ', name).strip()
    
    # 2. Xóa các ký tự icon/emoji (dải Unicode phổ biến cho emoji)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    cleaned_name = emoji_pattern.sub(r'', cleaned_name)
    
    return cleaned_name.strip()

if 'Tên sản phẩm' in df_sp.columns:
    print("Tên sản phẩm (trước khi xử lý):")
    print(df_sp['Tên sản phẩm'].head())
    
    df_sp['Tên sản phẩm'] = df_sp['Tên sản phẩm'].apply(clean_product_name)
    
    print("\nTên sản phẩm (sau khi xử lý):")
    print(df_sp['Tên sản phẩm'].head())

    # Ghi đè lại file
    save_path = input_path
    try:
        df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
    except PermissionError:
        save_path = input_path.with_name('sanpham_cleaned_final.csv')
        df_sp.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f'\nFile {input_path.name} đang bị khóa, đã lưu sang {save_path.name}')
    
    print(f'\nĐã cập nhật file: {save_path}')
else:
    print(f'Không tìm thấy cột "Tên sản phẩm" trong file {input_path.name}')

Tên sản phẩm (trước khi xử lý):
0    🔥100% hàng thật🔥La Roche-Posay Cicaplast Baume...
1    Bộ Chăm Sóc Da Hoa Mận LAIKOU Nhật Bản - 5 Sản...
2    Pores serum Thu Nhỏ Lỗ Chân Lông Úc Tinh Chất ...
3    Bộ chăm sóc da VIBRANT GLAMOUR ANNIES với tone...
4    Serum meiduzi x2 tem vuông phục hồi cấp ẩm - S...
Name: Tên sản phẩm, dtype: object

Tên sản phẩm (sau khi xử lý):
0    100% hàng thậtLa Roche-Posay Cicaplast Baume B...
1    Bộ Chăm Sóc Da Hoa Mận LAIKOU Nhật Bản - 5 Sản...
2    Pores serum Thu Nhỏ Lỗ Chân Lông Úc Tinh Chất ...
3    Bộ chăm sóc da VIBRANT GLAMOUR ANNIES với tone...
4    Serum meiduzi x2 tem vuông phục hồi cấp ẩm - S...
Name: Tên sản phẩm, dtype: object

Đã cập nhật file: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\sanpham_cleaned.csv


In [35]:
# Liệt kê kiểu dữ liệu của các cột trong file sanpham_cleaned.csv
from pathlib import Path
import pandas as pd

output_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/sanpham_cleaned.csv')
if output_path.exists():
    df_info = pd.read_csv(output_path)
    print(f"Kiểu dữ liệu của các cột trong file: {output_path.name}\n")
    df_info.info()
else:
    print(f"Không tìm thấy file sanpham_cleaned.csv để kiểm tra.")

Kiểu dữ liệu của các cột trong file: sanpham_cleaned.csv

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7027 entries, 0 to 7026
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID Sản phẩm    7027 non-null   int64  
 1   Tên sản phẩm   7027 non-null   object 
 2   Danh mục       7027 non-null   object 
 3   Giá            7027 non-null   float64
 4   Giảm giá       5347 non-null   float64
 5   Đã bán         7027 non-null   int64  
 6   Điểm đánh giá  5561 non-null   float64
 7   Lượt đánh giá  7027 non-null   float64
 8   Địa điểm       7027 non-null   object 
 9   Đường dẫn      7027 non-null   object 
 10  Hình ảnh       7027 non-null   object 
 11  Tên shop       6725 non-null   object 
 12  Seller ID      6943 non-null   float64
dtypes: float64(5), int64(2), object(6)
memory usage: 713.8+ KB


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Sử dụng đường dẫn tương đối từ thư mục notebooks/preprocessing/ tới data/preprocessed/
output_path = Path('../../data/preprocessed/sanpham_cleaned.csv')

if output_path.exists():
    # Đọc lại file
    df = pd.read_csv(output_path)
    
    # Chuyển 'ID Sản phẩm' sang chuỗi
    df['ID Sản phẩm'] = df['ID Sản phẩm'].astype(str)
    
    # Chuyển 'Seller ID' sang chuỗi.
    # Sử dụng apply để: nếu không phải NaN -> ép về int (bỏ đuôi .0) rồi chuyển sang chuỗi. Nếu NaN -> giữ nguyên NaN.
    if 'Seller ID' in df.columns:
        df['Seller ID'] = df['Seller ID'].apply(lambda x: str(int(x)) if pd.notna(x) else np.nan)
    
    # Kiểm tra lại kiểu dữ liệu sau khi sửa
    print("Kiểu dữ liệu sau khi chuyển đổi:\n")
    df.info()
    
    # Ghi đè lại file csv
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu thay đổi vào file: {output_path.name}")
    except PermissionError:
        print(f"\nLỗi: File {output_path.name} đang được mở ở phần mềm khác (ví dụ: Excel). Hãy đóng file và chạy lại code.")    
    
    # Xem thử một vài dòng đầu của 2 cột này
    print("\nMột vài dòng dữ liệu:")
    print(df[['ID Sản phẩm', 'Seller ID']].head())
else:
    print(f"Không tìm thấy file {output_path} để kiểm tra.")

Kiểu dữ liệu sau khi chuyển đổi:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7027 entries, 0 to 7026
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID Sản phẩm    7027 non-null   object 
 1   Tên sản phẩm   7027 non-null   object 
 2   Danh mục       7027 non-null   object 
 3   Giá            7027 non-null   float64
 4   Giảm giá       7027 non-null   float64
 5   Đã bán         7027 non-null   int64  
 6   Điểm đánh giá  5561 non-null   float64
 7   Lượt đánh giá  7027 non-null   float64
 8   Địa điểm       7027 non-null   object 
 9   Đường dẫn      7027 non-null   object 
 10  Hình ảnh       7027 non-null   object 
 11  Tên shop       6725 non-null   object 
 12  Seller ID      6943 non-null   object 
dtypes: float64(4), int64(1), object(8)
memory usage: 713.8+ KB

Đã lưu thay đổi vào file: sanpham_cleaned.csv

Một vài dòng dữ liệu:
  ID Sản phẩm     Seller ID
0  3140194631  200983952195
1  249664